# Comparison of Augmented Synthetic Control Method (ASCM)

This notebook compares the implementation of Augmented Synthetic Control in `causalis` and `pysyncon`.


This notebook presents the **ascm comparison** research workflow and key analysis steps.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from causalis.scenarios.synthetic_control.dgp import generate_scm_gamma_26
from causalis.data_contracts import PanelDataSCM
from causalis.scenarios.synthetic_control import AugmentedSyntheticControl
from pysyncon import Dataprep, AugSynth


## Data Generation

We use the `generate_scm_gamma_26` function to generate synthetic panel data.


In [2]:
df = generate_scm_gamma_26(
    return_panel_data=False,
    include_oracles=True,
)

# Ground-truth ATTE
post_treated = df[df["treated_time"] == 1].copy()
atte_true = post_treated["tau_realized_true"].mean()
print(f"Ground-truth ATTE is {atte_true:.6f}")
df.head()


Ground-truth ATTE is 1.919269


,unit_id,calendar_time,treated_time,y,y_cf,tau_realized_true,mu_cf,mu_treated,tau_mean_true
0,donor_1,2000-01,0,9.971611,9.971611,0.0,10.059081,10.059081,0.0
1,donor_1,2000-02,0,10.344379,10.344379,0.0,10.097467,10.097467,0.0
2,donor_1,2000-03,0,10.998498,10.998498,0.0,11.354098,11.354098,0.0
3,donor_1,2000-04,0,11.508717,11.508717,0.0,11.277716,11.277716,0.0
4,donor_1,2000-05,0,11.125281,11.125281,0.0,10.098791,10.098791,0.0


## CKit (causalis) ASCM

We use the `AugmentedSyntheticControl` model with `lambda_aug=50`.


In [3]:
paneldata = PanelDataSCM(
    df=df,
    y='y',
    unit_col='unit_id',
    time_col='calendar_time',
    treated_time='treated_time'
)

model_causalis = AugmentedSyntheticControl(lambda_aug=50).fit(paneldata)
result_causalis = model_causalis.estimate()
atte_causalis = float(result_causalis.diagnostics.get("average_att_estimate"))
print(f"Causalis ATTE: {atte_causalis:.6f}")
result_causalis.summary()


Causalis ATTE: 3.945634


,value
field,
estimand,average_post_effect
model,AugmentedSyntheticControl
inference,average_att_ttest
value,"3.9456 (ci_abs: 0.3664, 7.5249)"
value_relative,"16.0707 (ci_rel: 1.4922, 30.6491)"
alpha,0.0500
p_value,0.0417
is_significant,True
post_outcome_d_mean,28.3185


## PySyncon ASCM

We prepare the `Dataprep` object and fit the `AugSynth` model.


In [4]:
# Identify treated unit, controls, and time periods
treated_unit = df[df['treated_time'] == 1]['unit_id'].unique()[0]
control_units = df[df['unit_id'] != treated_unit]['unit_id'].unique().tolist()
# FIX: pre_periods should ONLY be periods where the TREATED unit is not yet treated.
# Previous code: pre_periods = sorted(df[df['treated_time'] == 0]['calendar_time'].unique().tolist())
# This incorrectly included periods where donors had treated_time=0 but the treated unit was already treated.
pre_periods = sorted(df[(df['unit_id'] == treated_unit) & (df['treated_time'] == 0)]['calendar_time'].unique().tolist())
post_periods = sorted(df[(df['unit_id'] == treated_unit) & (df['treated_time'] == 1)]['calendar_time'].unique().tolist())

dataprep = Dataprep(
    foo=df,
    predictors=[],
    predictors_op="mean",
    dependent="y",
    unit_variable="unit_id",
    time_variable="calendar_time",
    treatment_identifier=treated_unit,
    controls_identifier=control_units,
    time_predictors_prior=pre_periods,
    time_optimize_ssr=pre_periods,
)

model_pysyncon = AugSynth()
# We use lambda_=50 to be consistent with causalis's lambda_aug=50
model_pysyncon.fit(dataprep=dataprep, lambda_=50)

res_pysyncon = model_pysyncon.att(time_period=post_periods)
atte_pysyncon = res_pysyncon['att']
print(f"PySyncon ATTE: {atte_pysyncon:.6f}")


PySyncon ATTE: 3.766599


## Results Comparison


In [5]:
comparison = pd.DataFrame({
    "Method": ["Ground Truth", "Causalis", "PySyncon"],
    "ATTE": [atte_true, atte_causalis, atte_pysyncon]
})
comparison["Error"] = comparison["ATTE"] - atte_true
print(comparison)


         Method      ATTE     Error
0  Ground Truth  1.919269  0.000000
1      Causalis  3.945634  2.026365
2      PySyncon  3.766599  1.847330


### Explanation of Discrepancies

The estimates differ because:
1. **Time Period Alignment**: `causalis` automatically correctly identifies the pre-treatment period for the treated unit. In `pysyncon`, manually specifying `pre_periods` requires filtering by the treated unit to avoid including periods where donors are "pre-treatment" but the treated unit is already treated.
2. **De-meaning (Intercept)**: `pysyncon` performs row-wise (cross-sectional) de-meaning by the donor average at each time point before applying Ridge. This effectively includes a time-specific intercept $\alpha_t$. This also alters the direction of the Ridge penalty.
3. **Constraint Specification**: `causalis` explicitly enforces the sum-to-one constraint on augmented weights via a Lagrange multiplier in the original space, while `pysyncon` relies on cross-sectional de-meaning to satisfy the constraint (which is mathematically equivalent to solving with an intercept).
